In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
 
from fair import FAIR
from fair.interface import fill, initialise
from fair.io import read_properties

# FaIR analysis of CORSIA Phase 1 

The California analysis points out the way cross gas offsetting trade near-term cooling for long-term warming within an offsetting program. In California, that trade-off is most prominently driven by using credits representing methane abatement to justify ongoing CO2 emissions.

A similar dynamic is likely to arise in the context of the CORSIA program. CORSIA aims to offset any growth in internation aviation's CO2 emissions from 2024-2035. Airline / operators are the compliance entities. Any sector growth avoe the set basline (85% of 2019 CO2 emission levels) creates a CORSIA obligation (determined by participating member state governments and ICOA), which airlines must meet by purchasing and retiring eligible credits. 

The pilot phase of CORSIA (2021-2023) generated no offsetting obligations, seemingly because of the COVID-19 emissions drop ([slide 15](https://www.icao.int/sites/default/files/environmental-protection/CORSIA/Documents/CORSIA%20Periodic%20Review/CAEP_Inputs-to-2025-CORSIA-periodic-review-C234.pdf)). The first phase (2024-2026) is likely to generate some offsetting oblication — though it remains to be seen what the degree of compliance will be. 

There are a bunch of [rules](https://www.icao.int/sites/default/files/environmental-protection/CORSIA/Documents/CORSIA-Eligible-Emissions-Units_Oct2025.pdf) outlining which credits could become CORSIA eligible. However, today there are very few projects that have jumped through the hoops to obtain that eligibility stamp. A major barrier is that for this phase, project host countries must perform a corresponding adjustment, and to be labeled as CORSIA eligible, a project has to prove that will happen with an LOA from the host country. 

Given the uncertainty around which projects will actually become eligible, instead of doing a determinate analysis of the expected temperature impact of the program, it makes sense to simply explore potential scenarios for the gas fluxes underlying the credits used. 

# Offsetting obligation

In 2025, ICAO did a program review which presented updated scenarios for potential offsetting requirements in upcoming phases of the CORSIA program. See this [deck](https://www.icao.int/sites/default/files/environmental-protection/CORSIA/Documents/CORSIA%20Periodic%20Review/CAEP_Inputs-to-2025-CORSIA-periodic-review-C234.pdf), slide 31. We can use those scenarios to bound total volume of justified CO2 emissions and required offset credits.  

In [2]:
emission_years = [2024, 2025, 2026]
high_offsetting_requirement = [41, 49, 58] # million tCO2
low_offsetting_requirement = [29, 35, 37] # million tCO2

In [3]:
print(sum(high_offsetting_requirement))
print(sum(low_offsetting_requirement))

148
101


# Project fluxes

Credits that could be used to satisfy this offsetting obligation are heterogenous. The high level CORSIA eligibiliy rules designate programs, protocols, and eligibility windows (see [here](https://www.icao.int/sites/default/files/environmental-protection/CORSIA/Documents/CORSIA-Eligible-Emissions-Units_Oct2025.pdf), p.12-19). 

In [4]:
project_years = [2021, 2022, 2023, 2024, 2025, 2026] # CORSIA eligibility rules

Permanence periods and cross-gas equivalency metrics are set at the protocol level — and there's a lot of variability! Below, we define a couple of project archetypes that could be scaled up to meet CORSIA demand. 

- CO2 reduction; 20 year durability: [ART102](https://carbonplan.org/research/offsets-db/projects/ART102) is currently the largest certified eligible project. REDD credits are expected to provide a good portion of phase 1 credits (see e.g., [BeZero analysis](https://assets.bezerocarbonmarkets.com/f/179543/x/57861c5e56/bezero-corsia-market-review_-supply-integrity-demand-uncertainty-and-price-differentiation.pdf) Figure 3). ART REDD+ projects guarentee a 20-year durability period (see [Section 6.2](https://www.artredd.org/wp-content/uploads/2021/12/TREES-2.0-August-2021-Clean.pdf)). 
- CO2 reduction, no reversal: (e.g. cookstoves)
- CH4 reduction, GWP100: (e.g. landfills)
- CH4 reduction, GWP20: (TK: does anybody use GWP20 right now?) 
- Refrigerant destruction: (TK: how to deal with the mix of gases?)

In [5]:
# each of these archetypes represents one credit in terms of the underlying gas fluxes

co2_20y = np.full(100, 0.0)
co2_20y[0] = -1
co2_20y[21] = 1

co2_norev = np.full(100, 0.0)
co2_norev[0] = -1

ch4_100y = np.full(100, 0.0)
ch4_100y[0] = -1.0 / 28

ch4_20y = np.full(100, 0.0)
ch4_20y[0] = -1.0 / 81

In [6]:
# mix maps an archetype name to a fraction of the total offsetting obligation
# that is satisfies
#TODO: revisit blends; pull something from BeZero (fig 3) or sylvera? 
# https://assets.bezerocarbonmarkets.com/f/179543/x/57861c5e56/bezero-corsia-market-review_-supply-integrity-demand-uncertainty-and-price-differentiation.pdf
# https://info.sylvera.com/hubfs/Sylvera-CORSIA-First-Phase-Scenario-Modeling-Report-25.pdf?hsLang=en
# could also make a blend reflecting currently eligible projects 

mixes = {
    # corners
    "all_co2_20": {"co2_20y": 1.0},
    "all_co2_norev": {"co2_norev": 1.0},
    "all_ch4_gwp100": {"ch4_gwp100": 1.0},
    "all_ch4_gwp20": {"ch4_gwp20": 1.0},

    # blends
    "2025_eligibility": {
        "co2_20y": 0.77, # redd+
        "co2_norev": 0.23 # cookstoves + water
    },
    "bezero_estimate": {
        "co2_20y": 0.15, # redd+
        "co2_norev": 0.7, # household devices
        "ch4_100y":0.15, # waste + industrial processes
        "ch4_20y": 0
    }
}

In [7]:
# quick validator for mixes

def check_mixes(mixes):
    for name, mix in mixes.items():
        s = sum(mix.values())
        if not np.isclose(s, 1.0, atol=1e-6):
            raise ValueError(f"{name}: fractions sum to {s:.4f}, not 1.0")
    return True

assert check_mixes(mixes)


loop for each obligation (high/low)
--> credit array (even over project-years)
    for each mix vector 
    --> flux (scale archetype array by mix vector; place on master timeline; accumulate
    

In [ ]:
start_year = 2021
end_year = 2200
years = np.arange(start_year, end_year + 1)

co2_flux = np.zeros(len(years))
ch4_flux = np.zeros(len(years))

for requirement in (high_offsetting_requirement, low_offsetting_requirement):
    total_obligation = sum(requirement)
    credit_use_yearly = total_obligation / len(project_years)
    
    for archetype, fraction in mixes.items():
        archetype_yearly = credit_use_yearly * fraction
        
        # TO BE CONTINUED.... 

In [6]:
# low_volume = 32000000 #bezero estimate of currently eligible credits
# high_volume = 200000000 #high end estimate of demand 
#high_volume = 286000000 #bezero estimate of in-scope credits

101